#### Step 1: Load the data

In [1]:
import pandas as pd
import os

folder_path = "."
file_path = os.path.join(folder_path, "CSI300_merged_2005_2026.csv")

df = pd.read_csv(file_path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Keep only the columns used later in the workflow.
df = df[["date", "open", "high", "low", "close", "volume", "amount"]].copy()

print(df.head())
print(df.tail())
print(df.shape)

        date     open     high      low    close    volume     amount
0 2005-01-04  994.769  994.769  980.658  982.794  74128.69  443197.74
1 2005-01-05  981.577  997.323  979.877  992.564  71191.09  452920.82
2 2005-01-06  993.331  993.788  980.330  983.174  62880.29  392101.54
3 2005-01-07  983.045  995.711  979.812  983.958  72986.94  473746.94
4 2005-01-10  983.760  993.959  979.789  993.879  57916.98  376293.29
           date      open      high       low     close      volume  \
5152 2026-03-25  4506.625  4542.181  4502.027  4537.466  2626392.79   
5153 2026-03-26  4525.329  4541.050  4468.957  4477.534  2137367.88   
5154 2026-03-27  4431.503  4519.890  4431.421  4502.570  1924003.88   
5155 2026-03-30  4461.737  4496.411  4447.137  4491.950  2092930.92   
5156 2026-03-31  4491.921  4523.175  4450.049  4450.049  2245415.85   

           amount  
5152  55042816.12  
5153  47112749.84  
5154  44051756.16  
5155  44462951.97  
5156  48455811.30  
(5157, 7)


#### Step 2: Create the future 5-day return and the binary label

Definitions used in this notebook:

- `future_5d_ret`: return over the next 5 trading days
- `label_5d`: 1 if the future 5-day return is positive, otherwise 0


Step 2 reminder:

- `future_5d_ret`: return over the next 5 trading days
- `label_5d`: 1 if the future 5-day return is positive, otherwise 0


In [2]:
# Future 5-day return.
df["future_5d_ret"] = df["close"].shift(-5) / df["close"] - 1

# Binary label: 1 if the future 5-day return is positive, otherwise 0.
df["label_5d"] = (df["future_5d_ret"] > 0).astype(int)

print(df[["date", "close", "future_5d_ret", "label_5d"]].head(10))
print(df[["date", "close", "future_5d_ret", "label_5d"]].tail(10))

        date    close  future_5d_ret  label_5d
0 2005-01-04  982.794       0.014592         1
1 2005-01-05  992.564       0.004215         1
2 2005-01-06  983.174       0.013938         1
3 2005-01-07  983.958       0.004419         1
4 2005-01-10  993.879      -0.026590         0
5 2005-01-11  997.135      -0.022510         0
6 2005-01-12  996.748      -0.029634         0
7 2005-01-13  996.877      -0.040759         0
8 2005-01-14  988.306      -0.005769         0
9 2005-01-17  967.452       0.031713         1
           date     close  future_5d_ret  label_5d
5147 2026-03-18  4658.332      -0.025946         0
5148 2026-03-19  4583.251      -0.023066         0
5149 2026-03-20  4567.018      -0.014112         0
5150 2026-03-23  4417.997       0.016739         1
5151 2026-03-24  4474.722      -0.005514         0
5152 2026-03-25  4537.466            NaN         0
5153 2026-03-26  4477.534            NaN         0
5154 2026-03-27  4502.570            NaN         0
5155 2026-03-30  4491.95

#### Step 3: Reserve the valid sample range for a 20-day lookback window

At this stage we are not drawing images yet.  
We only keep rows that can serve as valid samples.

Because the model uses the past 20 trading days to predict the next 5 trading days:
- the first 19 rows cannot form a full lookback window,
- the last 5 rows cannot form a full forward label.


In [3]:
WINDOW = 20
HORIZON = 5

# A row is usable only if it has both a full past window and a full future label.
usable_df = df.iloc[WINDOW - 1 : len(df) - HORIZON].copy().reset_index(drop=True)

print("Number of usable samples:", len(usable_df))
print(usable_df.head())
print(usable_df.tail())

可用样本数: 5133
        date      open      high      low     close     volume      amount  \
0 2005-01-31   965.785   965.785  953.142   954.879   67887.06   386357.36   
1 2005-02-01   953.330   965.477  952.741   955.951   74330.67   427570.68   
2 2005-02-02   956.701  1006.932  956.701  1006.913  170573.39  1020290.43   
3 2005-02-03  1005.563  1014.187  992.155   993.215  169745.39  1005731.03   
4 2005-02-04   992.250  1021.025  989.939  1016.858  157362.15   954987.06   

   future_5d_ret  label_5d  
0       0.071952         1  
1       0.067634         1  
2      -0.000852         0  
3       0.032639         1  
4       0.029391         1  
           date      open      high       low     close      volume  \
5128 2026-03-18  4648.752  4662.294  4604.516  4658.332  2346907.05   
5129 2026-03-19  4611.440  4634.547  4570.392  4583.251  2541304.49   
5130 2026-03-20  4602.821  4628.712  4563.069  4567.018  2578737.03   
5131 2026-03-23  4499.100  4514.718  4397.566  4417.997  3280

#### Step 4: Build the rolling-window dataset

This step extracts the past 20-day OHLC window for every sample.

Once the data are stored as arrays, the same dataset can later be used for:
- direct input into traditional machine learning models,
- image rendering,
- CNN / MLP models.


In [4]:
import numpy as np

WINDOW = 20
feature_cols = ["open", "high", "low", "close"]

X = []
y = []
sample_dates = []

for i in range(WINDOW - 1, len(df) - HORIZON):
    window_data = df.loc[i - WINDOW + 1 : i, feature_cols].values   # Past 20-day OHLC window
    label = df.loc[i, "label_5d"]                                   # Future 5-day label aligned with date i
    sample_date = df.loc[i, "date"]

    X.append(window_data)
    y.append(label)
    sample_dates.append(sample_date)

X = np.array(X)   # shape: (n_samples, 20, 4)
y = np.array(y)   # shape: (n_samples,)
sample_dates = np.array(sample_dates)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("First 3 sample dates:", sample_dates[:3])
print("First 3 labels:", y[:3])

X shape: (5133, 20, 4)
y shape: (5133,)
前3个样本日期: [Timestamp('2005-01-31 00:00:00') Timestamp('2005-02-01 00:00:00')
 Timestamp('2005-02-02 00:00:00')]
前3个标签: [1 1 0]


In [5]:
save_path = os.path.join(folder_path, "CSI300_window20_label5d.npz")

np.savez(
    save_path,
    X=X,
    y=y,
    dates=sample_dates.astype(str)
)

print("Saved to:", save_path)

保存完成: /Users/daijinyang/Desktop/HS300/CSI300_window20_label5d.npz
